# 01. HCP Conversion

Build a file-map from `hcp_filt` and convert to `.pt` tensors.

This reproduces the nested output layout under:
`tensors/**/hcp_filt/<subject>/T1w/{rawavg.pt, aparc+aseg.pt}`.

In [1]:
from glob import glob
from pathlib import Path
from tqdm.auto import tqdm
from contextlib import contextmanager

import numpy as np
import pandas as pd

from joblib import Parallel, delayed
from joblib.parallel import BatchCompletionCallBack

import nibabel as nib
from nibabel.freesurfer.mghformat import MGHImage
from nilearn.image import load_img, resample_to_img

from scalesurfer.config import DATA_PATH
from scalesurfer.openneuro import list_all_files, convert_file_map_to_tensors

In [2]:
# Translate HCP images to 256^3 space
REF_ORIG = Path("/home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds000001__sub-13__ses-none__5698b04cd899/mri/orig.mgz")
REF_APARC_ASEG = Path("/home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds000001__sub-13__ses-none__5698b04cd899/mri/aparc+aseg.mgz")

@contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_cb = BatchCompletionCallBack
    try:
        import joblib.parallel
        joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_cb
        tqdm_object.close()


def _save_as_mgz(img, out_file: str | Path, dtype) -> None:
    data = np.asarray(img.get_fdata(), dtype=dtype)
    nib.save(MGHImage(data, img.affine), str(out_file))


def _process_subject(sub: str) -> None:
    sub = Path(sub)

    f_anat = sub / "T1w" / "T1w_acpc_dc_restore.nii.gz"
    f_anat_out = sub / "T1w" / "orig.mgz"
    assert f_anat.exists(), f"Missing: {f_anat}"

    anat_rs = resample_to_img(
        load_img(f_anat),
        load_img(REF_ORIG),
        interpolation="continuous",
        force_resample=True,
        copy_header=True,
    )
    _save_as_mgz(anat_rs, f_anat_out, np.float32)

    f_aparc_aseg = sub / "T1w" / "aparc+aseg.nii.gz"
    f_aparc_aseg_out = sub / "T1w" / "aparc+aseg.mgz"
    assert f_aparc_aseg.exists(), f"Missing: {f_aparc_aseg}"

    aparc_rs = resample_to_img(
        load_img(f_aparc_aseg),
        load_img(REF_APARC_ASEG),
        interpolation="nearest",
        force_resample=True,
        copy_header=True,
    )
    _save_as_mgz(aparc_rs, f_aparc_aseg_out, np.int32)


def run_parallel(n_jobs: int = -1) -> None:
    sub_dirs = glob("/home/rph/scalesurfer/data/hcp_filt/*")
    with tqdm_joblib(tqdm(total=len(sub_dirs), desc="Processing subjects")):
        Parallel(n_jobs=n_jobs)(delayed(_process_subject)(sub) for sub in sub_dirs)

run_parallel(16)

Processing subjects:   0%|          | 0/873 [00:00<?, ?it/s]

In [3]:
HCP_ROOT = DATA_PATH / "hcp_filt"
OUT_ROOT = DATA_PATH / "tensors"
N_JOBS = -1

if not HCP_ROOT.exists():
    raise FileNotFoundError(f"Missing HCP root: {HCP_ROOT.resolve()}")

files = list_all_files(HCP_ROOT)
files = [
    f
    for f in files
    if "T1w/" in f and (f.endswith("orig.mgz") or f.endswith("aparc+aseg.mgz"))
]

file_map_hcp = {}
for f in files:
    parts = f.split("/")
    key = "/".join(parts[:-1])
    name = parts[-1].split(".")[0]

    if key not in file_map_hcp:
        file_map_hcp[key] = {"orig": None, "aparc+aseg": None}

    if name == "T1w_acpc_dc_restore":
        name = "orig"
    file_map_hcp[key][name] = f

file_map_hcp_complete = {
    k: v for k, v in file_map_hcp.items()
    if v["orig"] is not None and v["aparc+aseg"] is not None
}

display(pd.DataFrame([{
    "hcp_root": str(HCP_ROOT.resolve()),
    "out_root": str(OUT_ROOT.resolve()),
    "pairs_found": len(file_map_hcp),
    "pairs_complete": len(file_map_hcp_complete),
}]))


,hcp_root,out_root,pairs_found,pairs_complete
0,/home/rph/convolutional_ar/segmentation/data/h...,/home/rph/scalesurfer/data/tensors,873,873


In [4]:
results = convert_file_map_to_tensors(
    file_map_hcp_complete,
    out_root=str(OUT_ROOT),
    n_jobs=N_JOBS,
)

ok = sum(1 for r in results if bool(r.get("ok", False)))
failed = len(results) - ok

display(pd.DataFrame([{
    "converted_total": len(results),
    "converted_ok": ok,
    "converted_failed": failed,
}]))


Converting:   0%|          | 0/873 [00:00<?, ?it/s]

,converted_total,converted_ok,converted_failed
0,873,873,0
